## Set Up the Environment

In [ ]:
%run setup.ipynb

## Document Loaders

Document loaders are used to import data from various sources into LangChain as `Document` objects. A `Document` typically includes a piece of text along with its associated metadata.

### Examples of Document Loaders:

- **Text File Loader:** Loads data from a simple `.txt` file.
- **Web Page Loader:** Retrieves the text content from any web page.
- **YouTube Video Transcript Loader:** Loads transcripts from YouTube videos.

### Functionality:

- **Load Method:** Each document loader has a `load` method that enables the loading of data as documents from a pre-configured source.
- **Lazy Load Option:** Some loaders also support a "lazy load" feature, which allows data to be loaded into memory gradually as needed.

For more detailed information, visit [LangChain's document loader documentation](https://python.langchain.com/docs/modules/data_connection/document_loaders/).


### PDF Loaders

[Portable Document Format (PDF)](https://en.wikipedia.org/wiki/PDF), standardized as ISO 32000, is a file format developed by Adobe in 1992 to present documents, including text formatting and images, in a manner independent of application software, hardware, and operating systems.

LangChain integrates with a host of PDF parsers. Some are simple and relatively low-level; others will support OCR and image-processing, or perform advanced document layout analysis. The right choice will depend on your use-case and through experimentation.

Here we will see how to load PDF documents into the LangChain `Document` format

We download a research paper to experiment with

If the following command fails you can download the paper manually by going to http://arxiv.org/pdf/2103.15348.pdf, save it as `layoutparser_paper.pdf`and upload it on the left in Colab from the upload files option

### PyPDFLoader

Here we load a PDF using `pypdf` into list of documents, where each document contains the page content and metadata with page number. Typically each PDF page becomes one document

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../../docs/layoutparser_paper.pdf")
pages = loader.load()

print(pages[0].page_content)
print(pages[0].metadata)

In [ ]:
len(pages)

In [ ]:
from pprint import pprint

In [ ]:
pprint(pages[0])

In [ ]:
pprint(pages[0].page_content)

In [ ]:
print(pages[0].metadata)

### PyMuPDFLoader

This is the fastest of the PDF parsing options, and contains detailed metadata about the PDF and its pages, as well as returns one document per page. It uses the `pymupdf` library internally.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../../docs/layoutparser_paper.pdf")
pages = loader.load()

print(pages[0].page_content)
print(pages[0].metadata)

In [ ]:
len(pages)

In [ ]:
print(pages[0])

In [ ]:
pages[0].metadata

In [ ]:
print(pages[0].page_content)

### UnstructuredPDFLoader


[Unstructured.io](https://unstructured-io.github.io/unstructured/) supports a common interface for working with unstructured or semi-structured file formats, such as Markdown or PDF. LangChain's [`UnstructuredPDFLoader`](https://api.python.langchain.com/en/latest/document_loaders/langchain_community.document_loaders.pdf.UnstructuredPDFLoader.html) integrates with Unstructured to parse PDF documents into LangChain [`Document`](https://api.python.langchain.com/en/latest/documents/langchain_core.documents.base.Document.html) objects.

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

loader = UnstructuredPDFLoader('../../docs/layoutparser_paper.pdf')
data = loader.load()

print(data[0].page_content)
print(data[0].metadata)

Load PDF with complex parsing, table detection and chunking by sections

Refer to https://community.databricks.com/t5/data-engineering/trying-to-use-pdf2image-on-databricks/td-p/12914


In [ ]:
# #install poppler on the cluster (should be done by init scripts)
# def install_ocr_on_nodes():
#     """
#     install poppler on the cluster (should be done by init scripts)
#     """
#     # from pyspark.sql import SparkSession
#     import subprocess
#     num_workers = max(1,int(spark.conf.get("spark.databricks.clusterUsageTags.clusterWorkers")))
#     command = "sudo rm -rf /var/cache/apt/archives/* /var/lib/apt/lists/* && sudo apt-get clean && sudo apt-get update && sudo apt-get install poppler-utils tesseract-ocr -y" 
#     def run_subprocess(command):
#         try:
#             output = subprocess.check_output(command, stderr=subprocess.STDOUT, shell=True)
#             return output.decode()
#         except subprocess.CalledProcessError as e:
#             raise Exception("An error occurred installing OCR libs:"+ e.output.decode())
#     #install on the driver
#     run_subprocess(command)
#     def run_command(iterator):
#         for x in iterator:
#             yield run_subprocess(command)
#     # spark = SparkSession.builder.getOrCreate()
#     data = spark.sparkContext.parallelize(range(num_workers), num_workers) 
#     # Use mapPartitions to run command in each partition (worker)
#     output = data.mapPartitions(run_command)
#     try:
#         output.collect();
#         print("OCR libraries installed")
#     except Exception as e:
#         print(f"Couldn't install on all node: {e}")
#         raise e

In [ ]:
# install_ocr_on_nodes()

In [ ]:
# takes 3-4 mins on Colab
loader = UnstructuredPDFLoader('../../docs/layoutparser_paper.pdf',
                               strategy='hi_res',
                               extract_images_in_pdf=False,
                               infer_table_structure=True,
                               chunking_strategy="by_title",
                               max_characters=4000, # max size of chunks
                               new_after_n_chars=3800, # preferred size of chunks
                               combine_text_under_n_chars=2000, # smaller chunks < 2000 chars will be combined into a larger chunk
                               mode='elements')
data = loader.load()

In [ ]:
data

In [ ]:
len(data)

In [ ]:
[doc.metadata['category'] for doc in data]

In [ ]:
pprint(data[0])

In [ ]:
print(data[0].page_content)

In [ ]:
print(data[5].metadata)

In [ ]:
from IPython.display import HTML

HTML(data[5].metadata['text_as_html'])

Load using raw unstructured.io APIs for PDFs

In [ ]:
from unstructured.partition.pdf import partition_pdf

# Get elements - takes 3-4 mins
raw_pdf_elements = partition_pdf(
    filename="./docs/layoutparser_paper.pdf",
    strategy='hi_res',
    # Unstructured first finds embedded image blocks
    extract_images_in_pdf=False,
    # Use layout model (YOLOX) to get bounding boxes (for tables) and find titles
    # Titles are any sub-section of the document
    infer_table_structure=True,
    # Post processing to aggregate text once we have the title
    chunking_strategy="by_title",
    # Chunking params to aggregate text blocks
    # Attempt to create a new chunk 3800 chars
    # Attempt to keep chunks > 2000 chars
    max_characters=4000,
    new_after_n_chars=3800,
    combine_text_under_n_chars=2000,
    image_output_dir_path="./",
)

In [ ]:
len(raw_pdf_elements)

In [ ]:
raw_pdf_elements

In [ ]:
raw_pdf_elements[5].to_dict()

Convert into LangChain `document`format

In [ ]:
from langchain_core.documents import Document

lc_docs = [Document(page_content=doc.text,
                    metadata=doc.metadata.to_dict())
              for doc in raw_pdf_elements]
lc_docs[5]